# Expected Free Energy Navigation: Point-Mass vs Differential-Drive

This notebook compares the two action spaces directly using the **original Julia cart2polar observation**:

\[
 g(x) = [\sqrt{x^2+y^2},\ \operatorname{atan2}(y,x)]
\]

So the comparison isolates dynamics/action-space effects under the same observation geometry and similar benchmark settings.

### Joost Leliveld, update: 02-03-2026

### Use case

- Same observation function and preference model for both planners.
- Same time settings and planning horizon.
- Compare behavior for:
  - point-mass control space `u=[u_x,u_y]`
  - differential-drive control space `u=[v,\omega]`

### Dynamics

Point-mass: linear constant-velocity model (`FieldBot`, `EFEAgent`).

Differential-drive: unicycle kinematics (`UnicycleBot`, `UnicycleEFEAgent`).

### Observations (Julia cart2polar)

Observation for both models depends only on world position `(x,y)`:

\[
 y = [r,\phi] = [\sqrt{x^2+y^2},\ \operatorname{atan2}(y,x)]
\]

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd()
ROOT = CWD if (CWD / "scripts").exists() else CWD.parent
sys.path.insert(0, str(ROOT / "scripts"))
ANIM_DIR = ROOT / "scripts" / "animations"
ANIM_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = ROOT / "scripts" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.optimize import minimize
from scipy.stats import multivariate_normal
from tqdm import tqdm

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

from robots import FieldBot, UnicycleBot, update, update_unicycle
from botnav_efe_helpers import (
    EFEAgent,
    UnicycleEFEAgent,
    predict,
    predict_unicycle,
    correct,
    evidence,
    planned_trajectory,
    planned_trajectory_unicycle,
)
from botnav_efe_jax import bind_agent_jax, bind_unicycle_agent_jax, make_valgrad_fn, make_unicycle_valgrad_fn

In [ ]:
# Matplotlib settings
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['lines.markersize'] = 3


def plot_cov_ellipse(ax, mean, cov, n_std=1, color='blue', alpha=0.25, linewidth=0):
    cov = np.asarray(cov, dtype=float)
    cov = 0.5 * (cov + cov.T)
    eigenvals, eigenvecs = np.linalg.eigh(cov)
    eigenvals = np.clip(eigenvals, 1e-12, None)
    order = eigenvals.argsort()[::-1]
    eigenvals, eigenvecs = eigenvals[order], eigenvecs[:, order]

    angle = np.degrees(np.arctan2(*eigenvecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(eigenvals)

    ell = Ellipse(xy=mean, width=width, height=height, angle=angle,
                  color=color, alpha=alpha, linewidth=linewidth)
    ax.add_patch(ell)


def g_cart2polar_xy(x, y):
    r = np.sqrt(x * x + y * y)
    phi = np.arctan2(y, x)
    return np.array([r, phi], dtype=float)


def g_pointmass(z):
    z = np.asarray(z, dtype=float)
    return g_cart2polar_xy(z[0], z[1])


def g_unicycle(z):
    z = np.asarray(z, dtype=float)
    return g_cart2polar_xy(z[0], z[1])


def g_pointmass_jax(z):
    z = jnp.asarray(z)
    r = jnp.sqrt(z[0] * z[0] + z[1] * z[1])
    phi = jnp.arctan2(z[1], z[0])
    return jnp.stack([r, phi])


def g_unicycle_jax(z):
    z = jnp.asarray(z)
    r = jnp.sqrt(z[0] * z[0] + z[1] * z[1])
    phi = jnp.arctan2(z[1], z[0])
    return jnp.stack([r, phi])

In [ ]:
# =========================
# EXPERIMENT SETUP (Julia-like)
# =========================

# --- Time ---
Δt = 0.2
len_trial = 20
len_horizon = 5

# --- Shared preference / noise settings ---
η = 0.0
σ_obs = 1e-3
ρ = np.array([1e-2, 1e-2])
goal_cov = 0.5 * np.eye(2)

# --- Shared XY start / goal from Julia cart2polar benchmark ---
start_xy = np.array([0.0, -0.5])
goal_xy = np.array([0.0, 0.5])

# Point-mass states (x,y,vx,vy)
z_0_pm = np.array([start_xy[0], start_xy[1], 0.0, 0.0])
z_star_pm = np.array([goal_xy[0], goal_xy[1], 0.0, 0.0])
goal_pm = (g_pointmass(z_star_pm), goal_cov)
m_0_pm = z_0_pm.copy()
S_0_pm = 0.5 * np.eye(4)
u_lims_pm = (-1.0, 1.0)

# Differential-drive states (x,y,theta)
theta_0 = np.arctan2(goal_xy[1] - start_xy[1], goal_xy[0] - start_xy[0])
z_0_uni = np.array([start_xy[0], start_xy[1], theta_0])
z_star_uni = np.array([goal_xy[0], goal_xy[1], theta_0])
goal_uni = (g_unicycle(z_star_uni), goal_cov)
m_0_uni = z_0_uni.copy()
S_0_uni = 0.5 * np.eye(3)

# Map Julia-like process scale to unicycle diagonal Q
sigma_process_xy = np.sqrt((Δt ** 3 / 3.0) * ρ[0])
sigma_process_theta = np.sqrt(Δt * ρ[0])
Q_uni = np.diag([sigma_process_xy**2, sigma_process_xy**2, sigma_process_theta**2])
R_uni = np.diag([σ_obs**2, σ_obs**2])

v_lims = (-1.0, 1.0)
w_lims = (-1.0, 1.0)

# Build bots/agents
fbot_pm = FieldBot(g_pointmass, ρ, σ=σ_obs, Δt=Δt, control_lims=u_lims_pm)
fbot_pm.R = np.diag([σ_obs**2, σ_obs**2])
agent_pm = EFEAgent(goal_pm, g_pointmass, ρ, sigma=σ_obs, eta=η, dt=Δt, time_horizon=len_horizon)

fbot_uni = UnicycleBot(g_unicycle, Q=Q_uni, R=R_uni, dt=Δt, control_lims=(v_lims, w_lims))
agent_uni = UnicycleEFEAgent(goal_uni, g_unicycle, Q=Q_uni, R=R_uni, eta=η, dt=Δt, time_horizon=len_horizon)

# JAX bindings
params_pm_j = bind_agent_jax(agent_pm)
params_uni_j = bind_unicycle_agent_jax(agent_uni)

assert agent_pm.time_horizon == len_horizon
assert params_pm_j.time_horizon == len_horizon
assert agent_uni.time_horizon == len_horizon
assert params_uni_j.time_horizon == len_horizon

print("Point-mass start/goal obs:", g_pointmass(z_0_pm), "->", g_pointmass(z_star_pm))
print("Diffdrive start/goal obs :", g_unicycle(z_0_uni), "->", g_unicycle(z_star_uni))
print("Q_uni diag:", np.diag(Q_uni))

## Model: EFE(ET2) single-run comparison

In [ ]:
def run_pointmass(seed=0, approx="ET2", add_ambiguity=True, maxiter=200, maxfun=2000, ftol=1e-6, grad_mode="fwd"):
    np.random.seed(seed)

    valgrad = make_valgrad_fn(params_pm_j, g_pointmass_jax, approx=approx, add_ambiguity=add_ambiguity, mode=grad_mode)

    z_est = (np.zeros((4, len_trial)), np.zeros((4, 4, len_trial)))
    z_pln = (np.zeros((len_trial, 4, len_horizon)), np.zeros((len_trial, 4, 4, len_horizon)))
    y_pln = (np.zeros((len_trial, agent_pm.Dy, len_horizon)), np.zeros((len_trial, agent_pm.Dy, agent_pm.Dy, len_horizon)))
    z_sim = np.zeros((4, len_trial))
    y_sim = np.zeros((agent_pm.Dy, len_trial))
    u_sim = np.zeros((2, len_trial))
    F = np.zeros(len_trial)
    J = np.zeros(len_trial)

    z_est[0][:, 0] = m_0_pm
    z_est[1][:, :, 0] = S_0_pm
    z_sim[:, 0] = z_0_pm

    m_kmin1 = m_0_pm
    S_kmin1 = S_0_pm

    policy = np.zeros((2, len_horizon))
    bounds = [u_lims_pm for _ in range(2 * len_horizon)]

    for k in range(1, len_trial):
        y_sim[:, k], z_sim[:, k] = update(fbot_pm, z_sim[:, k - 1], u_sim[:, k - 1])

        m_k_pred, S_k_pred = predict(agent_pm, m_kmin1, S_kmin1, u_sim[:, k - 1])
        m_k, S_k = correct(agent_pm, y_sim[:, k], m_k_pred, S_k_pred, approx=approx)

        F[k] = evidence(agent_pm, y_sim[:, k], m_k_pred, S_k_pred, approx=approx)
        J[k] = -multivariate_normal.logpdf(y_sim[:, k], goal_pm[0], goal_pm[1])

        z_est[0][:, k] = m_k
        z_est[1][:, :, k] = S_k

        def f(u):
            val, _ = valgrad(jnp.array(u), jnp.array(m_k), jnp.array(S_k))
            return float(val)

        def grad_u(u):
            _, grad = valgrad(jnp.array(u), jnp.array(m_k), jnp.array(S_k))
            return np.asarray(grad, dtype=float)

        x0 = np.zeros(2 * len_horizon) if k == 1 else np.concatenate([policy[:, 1:].T.reshape(-1), policy[:, -1]])
        result = minimize(
            f,
            x0,
            jac=grad_u,
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': maxiter, 'maxfun': maxfun, 'ftol': ftol},
        )

        policy = result.x.reshape((len_horizon, 2)).T
        if k == 1:
            assert policy.shape == (2, len_horizon)
        u_sim[:, k] = policy[:, 0]

        planned_states, planned_obs = planned_trajectory(agent_pm, policy, (m_k, S_k), approx=approx)
        z_pln[0][k, :, :] = planned_states[0]
        z_pln[1][k, :, :, :] = planned_states[1]
        y_pln[0][k, :, :] = planned_obs[0]
        y_pln[1][k, :, :, :] = planned_obs[1]

        m_kmin1 = m_k
        S_kmin1 = S_k

    return {
        "model": "pointmass",
        "z_sim": z_sim,
        "z_est": z_est,
        "z_pln": z_pln,
        "y_sim": y_sim,
        "u_sim": u_sim,
        "F": F,
        "J": J,
    }

In [ ]:
def run_diffdrive(seed=0, approx="ET2", add_ambiguity=True, maxiter=200, maxfun=2000, ftol=1e-6, grad_mode="fwd"):
    np.random.seed(seed)

    valgrad = make_unicycle_valgrad_fn(params_uni_j, g_unicycle_jax, approx=approx, add_ambiguity=add_ambiguity, mode=grad_mode)

    z_est = (np.zeros((3, len_trial)), np.zeros((3, 3, len_trial)))
    z_pln = (np.zeros((len_trial, 3, len_horizon)), np.zeros((len_trial, 3, 3, len_horizon)))
    y_pln = (np.zeros((len_trial, agent_uni.Dy, len_horizon)), np.zeros((len_trial, agent_uni.Dy, agent_uni.Dy, len_horizon)))
    z_sim = np.zeros((3, len_trial))
    y_sim = np.zeros((agent_uni.Dy, len_trial))
    u_sim = np.zeros((2, len_trial))
    F = np.zeros(len_trial)
    J = np.zeros(len_trial)

    z_est[0][:, 0] = m_0_uni
    z_est[1][:, :, 0] = S_0_uni
    z_sim[:, 0] = z_0_uni

    m_kmin1 = m_0_uni
    S_kmin1 = S_0_uni

    policy = np.zeros((2, len_horizon))
    bounds = [v_lims, w_lims] * len_horizon

    for k in range(1, len_trial):
        y_sim[:, k], z_sim[:, k] = update_unicycle(fbot_uni, z_sim[:, k - 1], u_sim[:, k - 1])

        m_k_pred, S_k_pred = predict_unicycle(agent_uni, m_kmin1, S_kmin1, u_sim[:, k - 1])
        m_k, S_k = correct(agent_uni, y_sim[:, k], m_k_pred, S_k_pred, approx=approx)

        F[k] = evidence(agent_uni, y_sim[:, k], m_k_pred, S_k_pred, approx=approx)
        J[k] = -multivariate_normal.logpdf(y_sim[:, k], goal_uni[0], goal_uni[1])

        z_est[0][:, k] = m_k
        z_est[1][:, :, k] = S_k

        def f(u):
            val, _ = valgrad(jnp.array(u), jnp.array(m_k), jnp.array(S_k))
            return float(val)

        def grad_u(u):
            _, grad = valgrad(jnp.array(u), jnp.array(m_k), jnp.array(S_k))
            return np.asarray(grad, dtype=float)

        x0 = np.zeros(2 * len_horizon) if k == 1 else np.concatenate([policy[:, 1:].T.reshape(-1), policy[:, -1]])
        result = minimize(
            f,
            x0,
            jac=grad_u,
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': maxiter, 'maxfun': maxfun, 'ftol': ftol},
        )

        policy = result.x.reshape((len_horizon, 2)).T
        if k == 1:
            assert policy.shape == (2, len_horizon)
        u_sim[:, k] = policy[:, 0]

        planned_states, planned_obs = planned_trajectory_unicycle(agent_uni, policy, (m_k, S_k), approx=approx)
        z_pln[0][k, :, :] = planned_states[0]
        z_pln[1][k, :, :, :] = planned_states[1]
        y_pln[0][k, :, :] = planned_obs[0]
        y_pln[1][k, :, :, :] = planned_obs[1]

        m_kmin1 = m_k
        S_kmin1 = S_k

    return {
        "model": "diffdrive",
        "z_sim": z_sim,
        "z_est": z_est,
        "z_pln": z_pln,
        "y_sim": y_sim,
        "u_sim": u_sim,
        "F": F,
        "J": J,
    }

In [ ]:
def range_contours(ax, center=(0.0, 0.0), radii=None):
    if radii is None:
        radii = [0.25, 0.5, 0.75, 1.0, 1.25]
    for rad in radii:
        ax.add_patch(plt.Circle(center, rad, fill=False, ls='--', lw=0.9, color='orange', alpha=0.25))


def compute_xy_limits(*runs, margin=0.15):
    xs = []
    ys = []
    for run in runs:
        xs.append(run["z_sim"][0, :])
        ys.append(run["z_sim"][1, :])
        xs.append(run["z_est"][0][0, :])
        ys.append(run["z_est"][0][1, :])
    xs.append(np.array([start_xy[0], goal_xy[0], 0.0]))
    ys.append(np.array([start_xy[1], goal_xy[1], 0.0]))

    x = np.concatenate(xs)
    y = np.concatenate(ys)
    xl = [x.min() - margin, x.max() + margin]
    yl = [y.min() - margin, y.max() + margin]
    return xl, yl


def plot_action_space_comparison(run_pm, run_uni, title="EFE2 (ET2 + ambiguity)"):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
    xl, yl = compute_xy_limits(run_pm, run_uni)

    for ax, run, label in zip(axes, [run_pm, run_uni], ["Point-Mass", "Differential-Drive"]):
        ax.set_title(label, loc='left', fontweight='bold')
        ax.set_xlim(xl)
        ax.set_ylim(yl)
        ax.set_xlabel("x [m]")
        ax.set_aspect("equal")
        ax.grid(alpha=0.2)
        range_contours(ax)

        ax.scatter([start_xy[0]], [start_xy[1]], color='green', marker='D', s=55, edgecolors='black', linewidths=0.4, label='start')
        ax.scatter([goal_xy[0]], [goal_xy[1]], color='red', s=55, edgecolors='black', linewidths=0.4, label='goal')

        ax.plot(run["z_sim"][0, :], run["z_sim"][1, :], color='blue', lw=2.2, label='true')
        ax.plot(run["z_est"][0][0, :], run["z_est"][0][1, :], color='purple', lw=2.0, label='inferred')

        for j in range(0, len_trial, 2):
            plot_cov_ellipse(ax, run["z_est"][0][0:2, j], run["z_est"][1][0:2, 0:2, j],
                             n_std=1, color='purple', alpha=0.08, linewidth=0)

    axes[0].set_ylabel("y [m]")
    handles, labels = axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    fig.legend(by_label.values(), by_label.keys(), loc='upper center', bbox_to_anchor=(0.5, 1.04), ncol=5, frameon=True)
    fig.suptitle(title, y=1.1)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()


def plot_control_comparison(run_pm, run_uni):
    t = np.arange(len_trial) * Δt
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

    axes[0].plot(t, run_pm["u_sim"][0, :], label='u_x', color='tab:blue')
    axes[0].plot(t, run_pm["u_sim"][1, :], label='u_y', color='tab:orange')
    axes[0].set_title("Point-Mass controls")
    axes[0].set_ylabel("control")
    axes[0].grid(alpha=0.2)
    axes[0].legend(loc='upper right')

    axes[1].plot(t, run_uni["u_sim"][0, :], label='v', color='tab:blue')
    axes[1].plot(t, run_uni["u_sim"][1, :], label='w', color='tab:orange')
    axes[1].set_title("Differential-Drive controls")
    axes[1].set_ylabel("control")
    axes[1].set_xlabel("time [s]")
    axes[1].grid(alpha=0.2)
    axes[1].legend(loc='upper right')

    fig.tight_layout()
    plt.show()

In [ ]:
run_pm = run_pointmass(seed=0, approx="ET2", add_ambiguity=True)
run_uni = run_diffdrive(seed=0, approx="ET2", add_ambiguity=True)

plot_action_space_comparison(run_pm, run_uni)
plot_control_comparison(run_pm, run_uni)

In [ ]:
def summarize_single_run(run, goal_xy):
    true_xy = run["z_sim"][:2, :].T
    est_xy = run["z_est"][0][:2, :].T

    final_dist = float(np.linalg.norm(true_xy[-1] - goal_xy))
    path_len = float(np.linalg.norm(np.diff(true_xy, axis=0), axis=1).sum())
    control_effort = float(np.sum(np.sum(run["u_sim"].T ** 2, axis=1)))
    mse = float(np.mean(np.sum((est_xy - true_xy) ** 2, axis=1)))

    return {
        "model": run["model"],
        "final_dist": final_dist,
        "path_len": path_len,
        "control_effort": control_effort,
        "mse": mse,
        "sum_F": float(np.sum(run["F"])),
        "sum_J": float(np.sum(run["J"])),
        "performance_F_plus_J": float(np.sum(run["F"]) + np.sum(run["J"])),
    }

rows = [summarize_single_run(run_pm, goal_xy), summarize_single_run(run_uni, goal_xy)]

try:
    import pandas as pd
    display(pd.DataFrame(rows).set_index("model").round(4))
except Exception:
    print(rows)

## Monte Carlo comparison (same observation model, different action spaces)

In [ ]:
def run_mc_models(n_runs=50, seed0=0, approx="ET2", add_ambiguity=True, use_sem=False,
                  maxiter=120, maxfun=1200, ftol=1e-6, grad_mode="fwd"):
    pm_true = np.zeros((n_runs, len_trial, 2))
    pm_est = np.zeros((n_runs, len_trial, 2))
    pm_u = np.zeros((n_runs, len_trial, 2))

    uni_true = np.zeros((n_runs, len_trial, 2))
    uni_est = np.zeros((n_runs, len_trial, 2))
    uni_u = np.zeros((n_runs, len_trial, 2))

    for r in tqdm(range(n_runs), desc="MC pointmass+diffdrive"):
        seed = seed0 + r
        rp = run_pointmass(seed=seed, approx=approx, add_ambiguity=add_ambiguity,
                           maxiter=maxiter, maxfun=maxfun, ftol=ftol, grad_mode=grad_mode)
        ru = run_diffdrive(seed=seed, approx=approx, add_ambiguity=add_ambiguity,
                           maxiter=maxiter, maxfun=maxfun, ftol=ftol, grad_mode=grad_mode)

        pm_true[r] = rp["z_sim"][:2, :].T
        pm_est[r] = rp["z_est"][0][:2, :].T
        pm_u[r] = rp["u_sim"].T

        uni_true[r] = ru["z_sim"][:2, :].T
        uni_est[r] = ru["z_est"][0][:2, :].T
        uni_u[r] = ru["u_sim"].T

    out = {
        "pm_true": pm_true,
        "pm_est": pm_est,
        "pm_u": pm_u,
        "uni_true": uni_true,
        "uni_est": uni_est,
        "uni_u": uni_u,
    }

    if use_sem:
        out["spread_mode"] = "SEM"
    else:
        out["spread_mode"] = "run-to-run std"

    return out


def plot_ribbon(ax, x_mean, y_mean, x_spread, y_spread, color, alpha=0.2):
    dx = np.gradient(x_mean)
    dy = np.gradient(y_mean)
    norm = np.sqrt(dx**2 + dy**2) + 1e-9
    nx = -dy / norm
    ny = dx / norm
    r = np.sqrt(x_spread**2 + y_spread**2)

    x_upper = x_mean + nx * r
    y_upper = y_mean + ny * r
    x_lower = x_mean - nx * r
    y_lower = y_mean - ny * r

    ax.fill(
        np.concatenate([x_upper, x_lower[::-1]]),
        np.concatenate([y_upper, y_lower[::-1]]),
        color=color,
        alpha=alpha,
        linewidth=0,
        zorder=1,
    )


def summarize_mc(true_runs, est_runs, u_runs, goal_xy, success_r=0.25):
    final_err = np.linalg.norm(true_runs[:, -1, :] - goal_xy.reshape(1, 2), axis=1)
    success = final_err < success_r
    path_len = np.linalg.norm(np.diff(true_runs, axis=1), axis=2).sum(axis=1)
    ctrl = np.sum(np.sum(u_runs ** 2, axis=2), axis=1)
    mse = np.mean(np.sum((est_runs - true_runs) ** 2, axis=2), axis=1)

    final_xy = true_runs[:, -1, :]
    if final_xy.shape[0] > 1:
        term_disp = float(np.sqrt(final_xy[:, 0].var(ddof=1) + final_xy[:, 1].var(ddof=1)))
    else:
        term_disp = 0.0

    return {
        "final_dist_mean": float(final_err.mean()),
        "final_dist_std": float(final_err.std(ddof=1)) if final_err.size > 1 else 0.0,
        "success_rate": float(success.mean()),
        "path_len_mean": float(path_len.mean()),
        "path_len_std": float(path_len.std(ddof=1)) if path_len.size > 1 else 0.0,
        "control_effort_mean": float(ctrl.mean()),
        "control_effort_std": float(ctrl.std(ddof=1)) if ctrl.size > 1 else 0.0,
        "terminal_dispersion": term_disp,
        "mse_mean": float(mse.mean()),
        "mse_std": float(mse.std(ddof=1)) if mse.size > 1 else 0.0,
    }

In [ ]:
n_runs = 50
seed0 = 0
use_sem = False

mc = run_mc_models(
    n_runs=n_runs,
    seed0=seed0,
    approx="ET2",
    add_ambiguity=True,
    use_sem=use_sem,
    maxiter=60,
    maxfun=600,
    ftol=1e-6,
    grad_mode="fwd",
)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.0), sharex=True, sharey=True)

for ax, title, true_runs, est_runs in [
    (axes[0], "Point-Mass", mc["pm_true"], mc["pm_est"]),
    (axes[1], "Differential-Drive", mc["uni_true"], mc["uni_est"]),
]:
    true_mean = true_runs.mean(axis=0)
    est_mean = est_runs.mean(axis=0)

    true_spread = true_runs.std(axis=0, ddof=1)
    est_spread = est_runs.std(axis=0, ddof=1)
    if use_sem:
        true_spread = true_spread / np.sqrt(n_runs)
        est_spread = est_spread / np.sqrt(n_runs)

    ax.set_title(title, loc='left', fontweight='bold')
    ax.set_aspect('equal')
    ax.set_xlabel("x [m]")
    ax.grid(alpha=0.2)
    range_contours(ax)

    plot_ribbon(ax, true_mean[:, 0], true_mean[:, 1], true_spread[:, 0], true_spread[:, 1], color='blue', alpha=0.22)
    plot_ribbon(ax, est_mean[:, 0], est_mean[:, 1], est_spread[:, 0], est_spread[:, 1], color='purple', alpha=0.16)

    ax.plot(true_mean[:, 0], true_mean[:, 1], color='blue', lw=2.2, label='true mean', zorder=3)
    ax.plot(est_mean[:, 0], est_mean[:, 1], color='purple', lw=2.0, label='inferred mean', zorder=3)

    ax.scatter([start_xy[0]], [start_xy[1]], color='green', marker='D', s=55,
               edgecolors='black', linewidths=0.4, label='start', zorder=5)
    ax.scatter([goal_xy[0]], [goal_xy[1]], color='red', s=55,
               edgecolors='black', linewidths=0.4, label='goal', zorder=5)

axes[0].set_ylabel("y [m]")

handles, labels = axes[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(by_label.values(), by_label.keys(), loc='upper center', bbox_to_anchor=(0.5, 1.03), ncol=4, frameon=True)
fig.suptitle(f"Monte Carlo ({n_runs} runs, ribbons = {'SEM' if use_sem else 'run-to-run std'})", y=1.08)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

rows = [
    {"model": "pointmass", **summarize_mc(mc["pm_true"], mc["pm_est"], mc["pm_u"], goal_xy)},
    {"model": "diffdrive", **summarize_mc(mc["uni_true"], mc["uni_est"], mc["uni_u"], goal_xy)},
]

try:
    import pandas as pd
    display(pd.DataFrame(rows).set_index("model").round(4))
except Exception:
    print(rows)